# Sentiment Analysis

Dataset: `scraped_articles_preprocessed.csv` 
Label sentimen: **Positif**, **Negatif**, **Netral**

---

## 1. Import Library

In [3]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS
import ast

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

print('Library berhasil diimport.')

Library berhasil diimport.


## 2. Load Dataset

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/MirzaFathi/Project-A---10/main/Data/scraped_articles_preprocessed.csv'

df = pd.read_csv(DATA_URL)

print('Jumlah data:', len(df))
print('Kolom:', df.columns.tolist())
df.head(3)

Jumlah data: 804
Kolom: ['title', 'source', 'link asli', 'published', 'content', 'category', 'sentimen', 'scrape_status', 'content_cleaned', 'tokenization', 'stopword_removed', 'stemming_output', 'stemming_cleaned']


## 3. Eksplorasi Data

Cek distribusi label sentimen dan kata-kata dominan per kelas sentimen.

In [ ]:
print('Distribusi Sentimen:')
print(df['sentimen'].value_counts())

In [ ]:
sentimen_counts = df['sentimen'].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(sentimen_counts.index, sentimen_counts.values, color=['#4CAF50', '#F44336', '#2196F3'])
plt.title('Distribusi Label Sentimen - TikTok Shop')
plt.xlabel('Sentimen')
plt.ylabel('Jumlah Artikel')

for i, v in enumerate(sentimen_counts.values):
    plt.text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = text.replace('e-commerce', 'ecommerce').replace('tiktok shop', 'tiktokshop')
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

stopwords_id = {
    'yang', 'dan', 'di', 'dengan', 'untuk', 'ini', 'itu', 'dari', 'ke', 'pada',
    'adalah', 'dalam', 'juga', 'tidak', 'oleh', 'ada', 'atau', 'akan', 'bisa',
    'sudah', 'serta', 'karena', 'sebagai', 'tersebut', 'telah', 'lebih', 'seperti',
    'mereka', 'kita', 'kami', 'saya', 'anda', 'ia', 'dia', 'atas', 'bawah',
    'setelah', 'sebelum', 'antara', 'para', 'dapat', 'bahwa', 'namun', 'namun',
    'jika', 'bila', 'maka', 'hingga', 'saat', 'ketika', 'masih', 'hanya', 'pun',
    'hal', 'bagi', 'agar', 'maupun', 'baik', 'lain', 'lainnya', 'secara', 'sangat',
    'atas', 'bisa', 'itu', 'ini', 'menjadi', 'sejak', 'tahun', 'persen', 'salah',
    'satu', 'dua', 'tiga', 'empat', 'lima', 'per', 'dan', 'atas', 'juta', 'miliar'
}

all_stopwords = STOPWORDS.union(stopwords_id)

df_wc = df.copy()
df_wc['content_cleaned'] = df_wc['content'].apply(clean_text)

sentimen_labels = ['Positif', 'Netral', 'Negatif']
wc_colors = ['Greens', 'Blues', 'Reds']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, label, cmap in zip(axes, sentimen_labels, wc_colors):
    teks_gabung = ' '.join(df_wc[df_wc['sentimen'] == label]['content_cleaned'].dropna().tolist())
    wc = WordCloud(
        width=600,
        height=400,
        background_color='white',
        colormap=cmap,
        max_words=100,
        stopwords=all_stopwords
    ).generate(teks_gabung)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'WordCloud - Sentimen {label}', fontsize=13, fontweight='bold')

plt.suptitle('WordCloud per Kelas Sentimen (content_cleaned + stopwords removed)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Persiapan Data

Kolom `content` berisi teks artikel mentah yang akan digunakan sebagai input model.  
Baris dengan nilai kosong pada `content` atau `sentimen` dihapus terlebih dahulu.

In [ ]:
df = df.dropna(subset=['content', 'sentimen']).reset_index(drop=True)

label_map = {'Negatif': 0, 'Netral': 1, 'Positif': 2}
df['label'] = df['sentimen'].map(label_map)

df = df.dropna(subset=['label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

print('Jumlah data setelah cleaning:', len(df))
print('Label mapping:', label_map)
print()
print('Contoh teks:')
print(df['content'].iloc[0][:300])

## 5. Split Data (Training & Testing)

Data dibagi 80% untuk training dan 20% untuk testing.  
Parameter `stratify=y` memastikan proporsi tiap kelas sama di kedua set.

In [ ]:
X = df['content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Data training : {len(X_train)} artikel')
print(f'Data testing  : {len(X_test)} artikel')

## 6. TF-IDF Vectorization

TF-IDF mengubah teks menjadi angka. Kata yang sering muncul di satu dokumen  
tapi jarang di dokumen lain akan mendapat bobot tinggi.

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Jumlah fitur TF-IDF: {X_train_tfidf.shape[1]}')

## 7. Model 1 - Naive Bayes

Naive Bayes adalah model klasik untuk klasifikasi teks. Model ini bekerja  
dengan menghitung probabilitas setiap kata muncul di tiap kelas sentimen.

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print(f'Akurasi Naive Bayes: {accuracy_score(y_test, y_pred_nb):.2%}')

In [ ]:
label_names = ['Negatif', 'Netral', 'Positif']

print('Classification Report - Naive Bayes')
print('=' * 50)
print(classification_report(y_test, y_pred_nb, target_names=label_names))

In [ ]:
cm_nb = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(6, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_nb, display_labels=label_names)
disp.plot(colorbar=False, cmap='Oranges')
plt.title('Confusion Matrix - Naive Bayes')
plt.tight_layout()
plt.show()

## 8. Model 2 - Logistic Regression

Logistic Regression adalah model yang belajar mencari batas pemisah antar kelas.  
Model ini sangat cocok digunakan bersama TF-IDF karena mampu menangani banyak fitur sekaligus.

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_test_tfidf)

print(f'Akurasi Logistic Regression: {accuracy_score(y_test, y_pred_lr):.2%}')

In [ ]:
print('Classification Report - Logistic Regression')
print('=' * 50)
print(classification_report(y_test, y_pred_lr, target_names=label_names))

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(6, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=label_names)
disp.plot(colorbar=False, cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.tight_layout()
plt.show()

## 9. Perbandingan Hasil

Bandingkan akurasi kedua model secara visual.

In [ ]:
acc_nb = accuracy_score(y_test, y_pred_nb)
acc_lr = accuracy_score(y_test, y_pred_lr)

models = ['Naive Bayes', 'Logistic Regression']
scores = [acc_nb, acc_lr]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, scores, color=['coral', 'steelblue'])
plt.ylim(0, 1.1)
plt.title('Perbandingan Akurasi Model - TikTok Shop')
plt.ylabel('Akurasi')

for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{score:.2%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Naive Bayes         : {acc_nb:.2%}')
print(f'Logistic Regression : {acc_lr:.2%}')

## 10. Analisis Sentimen per Portal Berita

Bagian ini menganalisis distribusi sentimen (Positif, Netral, Negatif) berdasarkan portal berita (`source`).  
Analisis ini membantu melihat apakah setiap portal memiliki kecenderungan framing pemberitaan yang berbeda-beda.

Tiga visualisasi disajikan:
1. **Stacked Bar Chart** — proporsi sentimen per portal
2. **Heatmap** — persentase sentimen per portal (normalized)
3. **Prediksi model terbaik (Naive Bayes)** — distribusi prediksi sentimen per portal

In [ ]:
# Pastikan kolom source tidak kosong
df_portal = df.dropna(subset=['source', 'sentimen']).copy()

# Ambil 8 portal dengan jumlah artikel terbanyak
top_portals = df_portal['source'].value_counts().head(8).index.tolist()
df_portal = df_portal[df_portal['source'].isin(top_portals)]

print('Portal yang dianalisis:')
print(df_portal['source'].value_counts())

In [ ]:
# Hitung jumlah artikel per (source, sentimen)
portal_sentimen = df_portal.groupby(['source', 'sentimen']).size().unstack(fill_value=0)

# Pastikan urutan kolom konsisten
for col in ['Positif', 'Netral', 'Negatif']:
    if col not in portal_sentimen.columns:
        portal_sentimen[col] = 0
portal_sentimen = portal_sentimen[['Positif', 'Netral', 'Negatif']]

# Urutkan portal berdasarkan jumlah artikel terbanyak
portal_sentimen = portal_sentimen.loc[top_portals]

# Stacked Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))

colors = {'Positif': '#4CAF50', 'Netral': '#2196F3', 'Negatif': '#F44336'}
bottom = [0] * len(portal_sentimen)

for sentimen in ['Positif', 'Netral', 'Negatif']:
    vals = portal_sentimen[sentimen].values
    bars = ax.bar(portal_sentimen.index, vals, bottom=bottom,
                  label=sentimen, color=colors[sentimen], edgecolor='white', linewidth=0.5)
    # Tambahkan label jumlah di tengah tiap segmen
    for i, (val, bot) in enumerate(zip(vals, bottom)):
        if val > 3:
            ax.text(i, bot + val / 2, str(val), ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')
    bottom = [b + v for b, v in zip(bottom, vals)]

ax.set_title('Distribusi Sentimen per Portal Berita (Top 8)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Portal Berita', fontsize=11)
ax.set_ylabel('Jumlah Artikel', fontsize=11)
ax.legend(title='Sentimen', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_xticklabels(portal_sentimen.index, rotation=30, ha='right')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap proporsi sentimen (persentase per portal)
portal_pct = portal_sentimen.div(portal_sentimen.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    portal_pct,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Persentase (%)'}
)
ax.set_title('Heatmap Proporsi Sentimen per Portal Berita (%)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Sentimen', fontsize=11)
ax.set_ylabel('Portal Berita', fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

print('\nProporsi Sentimen per Portal (%):')
print(portal_pct.round(1).to_string())

In [ ]:
# Prediksi sentimen seluruh dataset menggunakan Naive Bayes (model terbaik)
X_all_tfidf = tfidf.transform(df_portal['content'])
df_portal = df_portal.copy()
df_portal['pred_nb'] = nb_model.predict(X_all_tfidf)

inv_label_map = {0: 'Negatif', 1: 'Netral', 2: 'Positif'}
df_portal['pred_sentimen'] = df_portal['pred_nb'].map(inv_label_map)

# Hitung distribusi prediksi per portal
portal_pred = df_portal.groupby(['source', 'pred_sentimen']).size().unstack(fill_value=0)
for col in ['Positif', 'Netral', 'Negatif']:
    if col not in portal_pred.columns:
        portal_pred[col] = 0
portal_pred = portal_pred[['Positif', 'Netral', 'Negatif']]
portal_pred = portal_pred.loc[[p for p in top_portals if p in portal_pred.index]]

# Heatmap prediksi
portal_pred_pct = portal_pred.div(portal_pred.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    portal_pred_pct,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Persentase (%)'}
)
ax.set_title('Heatmap Prediksi Sentimen (Naive Bayes) per Portal Berita (%)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Sentimen', fontsize=11)
ax.set_ylabel('Portal Berita', fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

print('\nProporsi Prediksi Sentimen per Portal (%):')
print(portal_pred_pct.round(1).to_string())

In [ ]:
# Ringkasan insight: portal dengan sentimen dominan terbesar
print('=== INSIGHT: Sentimen Dominan per Portal (Label Asli) ===')
for portal in portal_sentimen.index:
    row = portal_pct.loc[portal]
    dominant = row.idxmax()
    pct = row.max()
    total = portal_sentimen.loc[portal].sum()
    print(f'{portal:<25} → Dominan: {dominant:<10} ({pct:.1f}%)  | Total artikel: {total}')